# Drive 연결

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


# 문제 1

[문제 1] - 베이징 PM2.5 정규성 검정 (Anderson-Darling)
환경부에서 베이징의 한 대기질 측정소에서 100일 동안 일평균 PM2.5 농도(μg/m³)를 측정하였다. Anderson-Darling 검정을 통해 PM2.5 농도가 정규분포를 따르는지 검정하고자 한다.

[데이터]
- 파일명: 6_3_1.csv
- 출처: UCI Machine Learning Repository - Beijing PM2.5 Data (2010-2014)
  - 원본: Liang et al. (2015)
  - 링크: https://archive.ics.uci.edu/dataset/381

[기간 정의]
- **시작일: 매년 3월 2일**
- **종료일: 시작일을 포함하여 연속 100일째 되는 날(포함)**
  - 예) 2012년 기준: 2012-03-02 ~ 2012-06-09 (총 100일)

[Data Description]
※ `6_3_1.csv`는 시간별(hourly) PM2.5 관측치로 구성되어 있다.
※ 문제 풀이 과정에서 날짜별로 평균을 계산하여 **일평균 PM2.5**를 만든 뒤, (1)~(3)을 수행한다.

[컬럼 설명]
- year : 연도(정수형)
- month : 월(정수형, 1~12)
- day : 일(정수형, 1~31)
- hour : 시간(정수형, 0~23)
- pm25 : PM2.5 농도(실수형, 단위: μg/m³)
- date : 날짜(문자형, YYYY-MM-DD)

[일평균 정의]
- 일평균 PM2.5 = 같은 date에 해당하는 pm25의 평균

[문항]
(1) 100일치 일평균 PM2.5 농도의 **평균**과 **표준편차**를 구하시오. (소수점 셋째 자리까지 반올림)
(2) 100일치 일평균 PM2.5가 정규분포를 따르는지 Anderson-Darling 검정을 실시하고, **검정통계량**을 계산하시오. (소수점 셋째 자리까지 반올림)
(3) (2)에서 얻은 p-value를 바탕으로 유의수준 5%에서 귀무가설의 **기각/채택** 여부를 결정하시오. (p-value는 소수점 셋째 자리까지 반올림)


## Anderson–Darling 정규성 검정 핵심 정리

- 목적 : 데이터가 **정규분포를 따른다고 볼 수 있는지** 검정

- 가설  
  + H0 : 정규분포를 따른다  
  + H1 : 정규분포를 따르지 않는다  

- 검정 통계량(stat)  
  + 정규분포에서의 **이탈 정도**를 수치화  
  + 값이 클수록 비정규성 큼  
  + 꼬리(tail) 영역 차이에 민감 → 극단값 탐지에 유리  

- p-value 해석 (α=0.05)  
  + p < 0.05 → H0 기각 → 정규분포 아님  
  + p ≥ 0.05 → H0 기각 못함 → 정규성 가정 가능  

- 주의사항  
  + 표본 많으면 작은 차이도 쉽게 기각  
  + 표본 너무 적으면 검정력 약함  
  + 히스토그램·QQ-plot과 함께 해석 권장  

- 본 문제 적용  
  + 시간별 PM2.5 → 날짜별 평균 → 일평균 100개 생성  
  + 100개 값에 A–D 정규성 검정 수행  
  + `normal_ad(x)` → (stat, p_value) 반환  

---

## 다른 정규성 검정과 비교

- Shapiro–Wilk  
  + 소표본에서 검정력 우수  
  + 대표본에서는 쉽게 기각 가능  

- Kolmogorov–Smirnov  
  + 모수 고정 시 적합  
  + 모수 추정 시 Lilliefors 검정 권장  
  + 꼬리 민감도는 A–D보다 약함  

---

## 요약
- 꼬리 민감도 : Anderson–Darling  
- 소표본 정규성 : Shapiro–Wilk  
- 모수 추정 포함 : Lilliefors(K–S 변형)  
- 표본 100개 수준 : A–D 또는 Shapiro–Wilk 사용 일반적


In [4]:
import numpy as np
import pandas as pd

# Anderson-Darling 정규성 검정(stat, p-value)
from statsmodels.stats.diagnostic import normal_ad

# Colab이면 드라이브 경로로 변경해서 사용 가능
DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Udemy/빅데이터분석기사_파이썬/작업형_제3유형/6회/data/6_3_1.csv"

print("[데이터 로드]")
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print(df.head(), "\n")

[데이터 로드]
shape: (8784, 6)
   year  month  day  hour   pm25        date
0  2012      1    1     0  275.0  2012-01-01
1  2012      1    1     1  303.0  2012-01-01
2  2012      1    1     2  215.0  2012-01-01
3  2012      1    1     3  222.0  2012-01-01
4  2012      1    1     4   85.0  2012-01-01 



# 문제 2

[문제 2] - 한 레스토랑에서 고객들의 팁(tip) 금액에 영향을 미치는 요인을 분석하기 위해 데이터를 수집하였다. 다음은 고객들의 식사 정보를 기록한 데이터이다.

[데이터]
- 파일명: 6_3_2.csv
- 출처: Seaborn Built-in Dataset
  - 원본: Bryant, P.G. and Smith, M (1995) "Practical Data Analysis: Case Studies in Business Statistics"
  - 링크: https://github.com/mwaskom/seaborn-data/blob/master/tips.csv

[Data Description]
※ `6_3_2.csv`는 는 한 레스토랑에서 수집된 244건의 식사 기록을 포함하며, 각 레코드는 고객이 지불한 총 식사 금액, 팁, 성별, 흡연 여부, 요일, 시간대, 일행 수 등의 정보를 담고 있다.

[컬럼 설명]
- total_bill: 총 식사 금액 (달러)
- tip: 팁 금액 (달러) - 종속변수
- sex: 성별 (Male/Female)
- smoker: 흡연 여부 (Yes/No)
- day: 요일 (Thur/Fri/Sat/Sun)
- time: 시간대 (Lunch/Dinner)
- size: 일행 수 (명)


[문항]
(1) 팁 금액(tip)을 종속 변수로 하고, 총 식사 금액(total_bill), 일행 수(size)를 독립 변수로 하는 다중회귀 분석을 수행하여 회귀 계수가 가장 높은 변수를 구하시오. (다중회귀모형 적합 시, 절편 포함)

(2) 유의수준 5% 하에서 각 독립 변수가 팁 금액에 미치는 영향이 통계적으로 유의미한지 판단하고, 유의미한 변수의 개수를 구하시오.

(3) 위에서 구축한 회귀모형의 결정계수(R²)를 구하시오. (소수점 셋째 자리까지 반올림)

In [5]:
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pathlib import Path

DATA_PATH = "/content/drive/MyDrive/Colab Notebooks/Udemy/빅데이터분석기사_파이썬/작업형_제3유형/6회/data/6_3_2.csv"

print("[데이터 로드]")
df = pd.read_csv(DATA_PATH)
print("shape:", df.shape)
print(df.head(), "\n")

[데이터 로드]
shape: (244, 7)
   total_bill   tip     sex smoker  day    time  size
0       16.99  1.01  Female     No  Sun  Dinner     2
1       10.34  1.66    Male     No  Sun  Dinner     3
2       21.01  3.50    Male     No  Sun  Dinner     3
3       23.68  3.31    Male     No  Sun  Dinner     2
4       24.59  3.61  Female     No  Sun  Dinner     4 

